# Compute microsam features for images

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
from image_processing_tools.util.load_files import find_files_by_pattern
from image_processing_tools.image_class.image_container import ImageContainer
from pathlib import Path
import matplotlib
# matplotlib.use('Agg') # Critical: prevents plotting to screen/RAM
from image_processing_tools.util.align_dic_dapi import calculate_dic_shift

search_path = search_path = (
    "~/A8/Data_Eva/Training data Chuyao/MONO culture in DMEM + 10% FBS/CET161_ASH1Q670_02_CY5, DAPI",
    "~/A8/Data_Eva/Training data Chuyao/MONO culture in DMEM + 10% FBS/CET155_ASH1Q670_04_CY5, DAPI",
    "~/A8/Data_Eva/Training data Chuyao/MONO culture in DMEM + 10% FBS/CET147_ECE1Q670_06_CY5, DAPI",
    "~/A8/Data_Eva/Training data Chuyao/MONO culture in DMEM + 10% FBS/CET146_ASH1Q670_07_CY5, DAPI",
    "~/A8/Data_Eva/Training data Chuyao/MONO culture in DMEM + 10% FBS/CET146_ASH1Q670_02_CY5, DAPI",
    "~/A8/Data_Eva/Training data Chuyao/MONO culture in DMEM + 10% FBS/CET145_ASH1Q670_02_CY5, DAPI",
    )
file_pattern = ("CROP_*.tif",
                "MAX_*",
                "SHARPEST*C1*.tif",
                "*DIC*.tif",
                "MAX_CET145*")

config = {
    "preprocessing": {
        "resize_image": False,
        "max_dim": 1080,
        "outlier_percentile": 0.35,
        "quantization": "8bit",
        "correct_DIC_shift" : [5,22] # [Y,X] more negative Y to shift up, more positive X to shift to right
    }
}

# Find the files. The 'files' variable will hold the list of Path objects.
dapi_files = []
dic_files = []
for p in search_path: 
    dapi_files.append(find_files_by_pattern(p, file_pattern[2], verbose=True))
    dic_files.append(find_files_by_pattern(p, file_pattern[3], verbose=True))

dapi_imgs = [ImageContainer(img,config) for img in dapi_files]
dic_imgs = [ImageContainer(img,config) for img in dic_files]

merged_imgs = [dapi+dic for dapi,dic in zip(dapi_imgs, dic_imgs)]

In [ ]:
merged_imgs[0].display()

In [ ]:
from image_processing_tools.dapi_tracing.precompute_microsam_feats import compute_microsam_features, _get_encoder

microsam_feat_config = {
    "model_type": "vit_l_lm",                                                                                                                                
    "checkpoint_path": Path("~/Projects/C_Albicans Thesis Project/7. Data/model_checkpoints/micro_sam/models/vit_l_lm").expanduser(),                                                                                                          
    "tile_shape": (512, 512),                                                                                                                                
    "halo": (64, 64),
}

predictor = _get_encoder(microsam_feat_config["model_type"],microsam_feat_config["checkpoint_path"])

for i,image in enumerate(merged_imgs):
    print(f"Processing image {i+1}...")
    _,_ = compute_microsam_features(image, microsam_feat_config, predictor=predictor)

In [ ]:
from micro_sam.util import get_device

get_device()